


dataset : https://www.kaggle.com/datasets/social888/vehicles/data


Imported libraries

In [ ]:
import os
import glob
import random
import shutil
import time
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, Model, Sequential
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU,
    MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from ultralytics import YOLO

Vehicle classification

In [ ]:

# =========================
# CONFIG
# =========================
DATA_DIR = "/kaggle/input/datasets/social888/vehicles/vehicleclass/vehicleclass"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# =========================
# VERIFY DATA STRUCTURE
# =========================
print("Checking dataset structure...")
for root, dirs, files in os.walk(DATA_DIR):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", files[:5])
    break

# =========================
# CLASS MAPPING
# =========================
class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

class_to_idx = {name: i for i, name in enumerate(class_names)}

# =========================
# LOAD FILE PATHS
# =========================
filepaths, labels = [], []

for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)

    for img in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img)

        # skip folders
        if not os.path.isfile(img_path):
            continue

        # keep only images
        if not img.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        filepaths.append(img_path)
        labels.append(class_to_idx[cls])

filepaths = np.array(filepaths)
labels = np.array(labels)

# =========================
# DEBUG CHECK
# =========================
print("\nTotal images:", len(filepaths))
print("Sample path:", filepaths[0])
print("Classes:", class_names)

labels = np.array(labels)

class_counts = Counter(labels)

for class_idx, count in class_counts.items():
    print(class_names[class_idx], ":", count)
# =========================
# SPLITS
# =========================
X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
    filepaths, labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp_raw, y_temp_raw,
    test_size=0.5,
    stratify=y_temp_raw,
    random_state=SEED
)


Checking dataset structure...
ROOT: /kaggle/input/datasets/social888/vehicles/vehicleclass/vehicleclass
DIRS: ['Motorcycle', 'Bus', 'truck', 'car']
FILES: []

Total images: 1773
Sample path: /kaggle/input/datasets/social888/vehicles/vehicleclass/vehicleclass/Bus/bus-72_jpg.rf.XYhCVPSsD54CUuVyY5pG.jpg
Classes: ['Bus', 'Motorcycle', 'car', 'truck']
Bus : 383
Motorcycle : 395
car : 587
truck : 408


Evaluation and plotting funtions

In [ ]:
def print_final_metrics(history, name):
    print(f"\n{name}")

    print("Final Train Accuracy:", history.history['accuracy'][-1])
    print("Final Val Accuracy:", history.history['val_accuracy'][-1])

    print("Final Train Loss:", history.history['loss'][-1])
    print("Final Val Loss:", history.history['val_loss'][-1])
    
def evaluate(model, dataset):
    y_true, y_pred = [], []

    for x, y in dataset:
        preds = model.predict(x, verbose=0)
        y_true.extend(y.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    print(classification_report(y_true, y_pred, target_names=class_names))
    return confusion_matrix(y_true, y_pred)

def plot(history, title):
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='val')
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.title(title + " Loss")
    plt.legend()
    plt.show()


In [ ]:
def evaluate(model, dataset):
    y_true, y_pred = [], []

    for x, y in dataset:
        preds = model.predict(x, verbose=0)

        y_true.extend(y.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    # Classification report
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names
    ))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,          # numbers inside boxes
        fmt='d',             # integer format
        cmap='Blues',        # color theme
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")

    plt.show()

    return cm

Phase 1 — Dataset & Baseline CNN

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


def preprocess_custom(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def make_dataset_custom(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(preprocess_custom, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds_custom = make_dataset_custom(X_train_raw, y_train_raw, True)
val_ds_custom   = make_dataset_custom(X_val_raw, y_val_raw, False)
test_ds_custom  = make_dataset_custom(X_test_raw, y_test_raw, False)

I0000 00:00:1779052139.771406      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779052139.777889      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:

num_classes = len(class_names)

def build_custom_cnn():
    model = Sequential([
    
    # Input
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Block 4
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Feature pooling (important fix)
    layers.GlobalAveragePooling2D(),

    # Classification head
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output layer
    layers.Dense(num_classes, activation='softmax')
])
    return model

In [7]:
custom_model = build_custom_cnn()

custom_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

start = time.time()

history_custom = custom_model.fit(
    train_ds_custom,
    validation_data=val_ds_custom,
    epochs=20,
    callbacks=[early_stop]
)

custom_time = time.time() - start

print(f"\nTraining time: {custom_time:.2f} seconds")

Epoch 1/20


I0000 00:00:1779052166.827004     132 service.cc:152] XLA service 0x7c19c0112510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779052166.827043     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779052166.827048     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779052167.376348     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


 2/39 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.2891 - loss: 1.3941

I0000 00:00:1779052174.055541     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


39/39 ━━━━━━━━━━━━━━━━━━━━ 19s 258ms/step - accuracy: 0.3032 - loss: 1.3857 - val_accuracy: 0.3759 - val_loss: 1.3316
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.3876 - loss: 1.3068 - val_accuracy: 0.4511 - val_loss: 1.2144
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.4459 - loss: 1.1706 - val_accuracy: 0.4887 - val_loss: 1.1432
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.4792 - loss: 1.1758 - val_accuracy: 0.5226 - val_loss: 1.1281
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.5270 - loss: 1.1293 - val_accuracy: 0.5677 - val_loss: 1.0862
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.5413 - loss: 1.0415 - val_accuracy: 0.5714 - val_loss: 1.0434
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.5980 - loss: 1.0445 - val_accuracy: 0.5940 - val_loss: 1.0671
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.5443 - loss: 1.0694 - val_accuracy: 0.6090 - val_loss: 

In [8]:
print_final_metrics(history_custom, "Custom CNN")


Custom CNN
Final Train Accuracy: 0.7510072588920593
Final Val Accuracy: 0.7368420958518982
Final Train Loss: 0.7069017291069031
Final Val Loss: 0.7436437010765076


In [ ]:


cm_custom = evaluate(custom_model, test_ds_custom)

              precision    recall  f1-score   support

         Bus       0.55      0.81      0.66        58
  Motorcycle       0.90      0.80      0.85        59
         car       0.77      0.82      0.79        88
       truck       0.80      0.46      0.58        61

    accuracy                           0.73       266
   macro avg       0.76      0.72      0.72       266
weighted avg       0.76      0.73      0.73       266



In [ ]:
plot(history_custom, "Custom CNN")


In [12]:
custom_loss, custom_acc = custom_model.evaluate(test_ds_custom, verbose=0)

Phase 2 — Transfer Learning & Model Improvement

In [ ]:


data_aug_mnv2 = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
])


def preprocess_train_mnv2(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)

    img = tf.cast(img, tf.float32)
    img = data_aug_mnv2(img)
    img = preprocess_input(img)

    return img, label


def preprocess_test_mnv2(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)

    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)

    return img, label


def make_dataset_mnv2(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        ds = ds.shuffle(len(X), seed=SEED)
        ds = ds.map(preprocess_train_mnv2, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(preprocess_test_mnv2, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds_mnv2 = make_dataset_mnv2(X_train_raw, y_train_raw, True)
val_ds_mnv2   = make_dataset_mnv2(X_val_raw, y_val_raw, False)
test_ds_mnv2  = make_dataset_mnv2(X_test_raw, y_test_raw, False)

In [ ]:

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

mnv2_model = Model(inputs, outputs)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
mnv2_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [16]:
# =========================
# PHASE 1: FROZEN TRAINING
# =========================

start_phase1 = time.time()

history_phase1 = mnv2_model.fit(
    train_ds_mnv2,
    validation_data=val_ds_mnv2,
    epochs=10,
    callbacks=[early_stop]
)

phase1_time = time.time() - start_phase1



Epoch 1/10


2026-05-17 21:11:56.608161: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-17 21:11:56.746086: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


38/39 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.4491 - loss: 1.3079

2026-05-17 21:12:13.676442: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-17 21:12:13.813778: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 449ms/step - accuracy: 0.4536 - loss: 1.2988

2026-05-17 21:12:28.718441: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-17 21:12:28.855768: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


39/39 ━━━━━━━━━━━━━━━━━━━━ 49s 837ms/step - accuracy: 0.4579 - loss: 1.2900 - val_accuracy: 0.9211 - val_loss: 0.3411
Epoch 2/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 177ms/step - accuracy: 0.8477 - loss: 0.4733 - val_accuracy: 0.9323 - val_loss: 0.2313
Epoch 3/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 175ms/step - accuracy: 0.8853 - loss: 0.3938 - val_accuracy: 0.9398 - val_loss: 0.1950
Epoch 4/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 177ms/step - accuracy: 0.8834 - loss: 0.3548 - val_accuracy: 0.9436 - val_loss: 0.1886
Epoch 5/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 183ms/step - accuracy: 0.9034 - loss: 0.2936 - val_accuracy: 0.9511 - val_loss: 0.1720
Epoch 6/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 179ms/step - accuracy: 0.9239 - loss: 0.2330 - val_accuracy: 0.9511 - val_loss: 0.1569
Epoch 7/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 175ms/step - accuracy: 0.9027 - loss: 0.2676 - val_accuracy: 0.9549 - val_loss: 0.1610
Epoch 8/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 173ms/step - accuracy: 0.9234 - loss: 0.2248 - val_accuracy: 0.9474 - val

In [ ]:
# =========================
# PHASE 1 EVALUATION
# =========================

print("\n===== PHASE 1 EVALUATION =====")
print_final_metrics(history_phase1, "MobileNetV2")
cm_phase1 = evaluate(mnv2_model, val_ds_mnv2)
plot(history_phase1, "MobileNetV2 Phase 1")
mnv2_p1_loss, mnv2_p1_acc = mnv2_model.evaluate(test_ds_mnv2, verbose=0)

In [19]:

# =========================
# PHASE 2: FINE-TUNING (UNFREEZE PARTIAL BACKBONE)
# =========================

base_model.trainable = True

fine_tune_at = 100  # last layers trainable

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True


mnv2_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================
# PHASE 2 TRAINING
# =========================

start_phase2 = time.time()

history_phase2 = mnv2_model.fit(
    train_ds_mnv2,
    validation_data=val_ds_mnv2,
    epochs=5,
    callbacks=[early_stop]
)

phase2_time = time.time() - start_phase2




Epoch 1/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 45s 667ms/step - accuracy: 0.8045 - loss: 0.5582 - val_accuracy: 0.9511 - val_loss: 0.1311
Epoch 2/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 203ms/step - accuracy: 0.8873 - loss: 0.3543 - val_accuracy: 0.9549 - val_loss: 0.1237
Epoch 3/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 198ms/step - accuracy: 0.9031 - loss: 0.3179 - val_accuracy: 0.9549 - val_loss: 0.1205
Epoch 4/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 187ms/step - accuracy: 0.9070 - loss: 0.3049 - val_accuracy: 0.9549 - val_loss: 0.1227
Epoch 5/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 188ms/step - accuracy: 0.9166 - loss: 0.2583 - val_accuracy: 0.9549 - val_loss: 0.1219


In [ ]:
# =========================
# PHASE 2 EVALUATION
# =========================

print("\n===== PHASE 2 EVALUATION =====")
print_final_metrics(history_phase2, "MobileNetV2")
cm_phase2 = evaluate(mnv2_model, val_ds_mnv2)
plot(history_phase2, "MobileNetV2 Phase 2")
mnv2_p2_loss, mnv2_p2_acc = mnv2_model.evaluate(test_ds_mnv2, verbose=0)

Evaluation of models

In [ ]:


# FINAL COMPARISON TABLE
# =========================
comparison = pd.DataFrame({
    "Model": [
        "Custom CNN",
        "MobileNetV2 (Frozen - Phase 1)",
        "MobileNetV2 (Fine-tuned - Phase 2)"
    ],

    "Test Accuracy": [
        custom_acc,
        mnv2_p1_acc,
        mnv2_p2_acc
    ],

    "Test Loss": [
        custom_loss,
        mnv2_p1_loss,
        mnv2_p2_loss
    ],

    "Training Time (s)": [
        custom_time,
        phase1_time,
        phase2_time
    ],

    "Parameters": [
        custom_model.count_params(),
        mnv2_model.count_params(),
        mnv2_model.count_params()
    ]
})

print(comparison)

                                Model  Test Accuracy  Test Loss  \
0                          Custom CNN       0.729323   0.801230   
1      MobileNetV2 (Frozen - Phase 1)       0.954887   0.143920   
2  MobileNetV2 (Fine-tuned - Phase 2)       0.966165   0.148405   

   Training Time (s)  Parameters  
0          65.570206      487620  
1         111.483623     2263108  
2          75.351952     2263108  


Phase 3 — Object Detection or Segmentation

Track A — Object Detection

In [25]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00a 0:00:01


In [ ]:


dataset_path = "/kaggle/input/datasets/social888/vehicles/Vehicles.yolov8"

# list top-level contents
for root, dirs, files in os.walk(dataset_path):
    print(root)
    break

/kaggle/input/datasets/social888/vehicles/Vehicles.yolov8


In [ ]:


src = "/kaggle/input/datasets/social888/vehicles/Vehicles.yolov8/Vehicles.yolov8"

img_src = os.path.join(src, "train/images")
lbl_src = os.path.join(src, "train/labels")

out = "/kaggle/working/dataset"

images = [f for f in os.listdir(img_src) if f.endswith((".jpg", ".png", ".jpeg"))]
random.seed(42)
random.shuffle(images)

train_split = int(0.8 * len(images))
val_split = int(0.9 * len(images))

splits = {
    "train": images[:train_split],
    "val": images[train_split:val_split],
    "test": images[val_split:]
}

for s in splits:
    os.makedirs(f"{out}/{s}/images", exist_ok=True)
    os.makedirs(f"{out}/{s}/labels", exist_ok=True)

    for img in splits[s]:
        label = img.rsplit(".", 1)[0] + ".txt"

        shutil.copy(f"{img_src}/{img}", f"{out}/{s}/images/{img}")

        if os.path.exists(f"{lbl_src}/{label}"):
            shutil.copy(f"{lbl_src}/{label}", f"{out}/{s}/labels/{label}")

In [23]:
yaml_path = "/kaggle/working/dataset/data.yaml"

yaml_content = f"""
train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images

nc: 4
names: ['Bus', 'Motorcycle', 'car', 'truck']
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)




In [ ]:


model = YOLO("yolov8s.pt")

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    save=True,
    project="/kaggle/working/runs",
    name="yolov8_custom"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

Evaluate using IoU, Precision, Recall, and mAP on a test set 

In [21]:



metrics = model.val(
    data=yaml_path,
    split="test"
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision per class:", metrics.box.p)
print("Recall per class:", metrics.box.r)

Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1089.8±631.4 MB/s, size: 37.8 KB)
val: Scanning /kaggle/working/dataset/test/labels... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153 1.2Kit/s 0.1s
val: New cache created: /kaggle/working/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.4it/s 2.9s
                   all        153        214      0.916      0.866      0.931      0.729
                   Bus         32         41      0.872      0.878       0.93      0.811
            Motorcycle         40         65      0.952      0.919      0.963      0.595
                   car         55         65      0.851      0.785      0.876      0.706
                 truck         39         43      0.987      0.884     

In [ ]:


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def yolo_to_xyxy(box, w, h):
    x, y, bw, bh = box
    return [
        (x - bw / 2) * w,
        (y - bh / 2) * h,
        (x + bw / 2) * w,
        (y + bh / 2) * h
    ]

test_images = "/kaggle/working/dataset/test/images"
test_labels = "/kaggle/working/dataset/test/labels"
ious = []

for img_path in glob.glob(test_images + "/*.jpg"):
    label_path = os.path.join(
        test_labels,
        os.path.basename(img_path).replace(".jpg", ".txt")
    )
    if not os.path.exists(label_path):
        continue

    results = model(img_path, conf=0.25, verbose=False)[0]
    h, w = results.orig_shape
    preds = results.boxes.xyxy.cpu().numpy()
    pred_cls = results.boxes.cls.cpu().numpy().astype(int)  # cast once

    gt_boxes, gt_cls = [], []
    with open(label_path) as f:
        for line in f:
            c, x, y, bw, bh = map(float, line.split())
            gt_boxes.append(yolo_to_xyxy([x, y, bw, bh], w, h))
            gt_cls.append(int(c))

    matched_gt = set()  # track which GT boxes have been matched

    # --- Match each prediction to the best available GT box ---
    for p_box, p_c in zip(preds, pred_cls):
        best_iou = 0
        best_gt_idx = -1

        for g_idx, (g_box, g_c) in enumerate(zip(gt_boxes, gt_cls)):
            if p_c != g_c or g_idx in matched_gt:
                continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g_idx

        if best_gt_idx >= 0:
            matched_gt.add(best_gt_idx)
            ious.append(best_iou)
        else:
            ious.append(0)  # false positive → IoU = 0

    # --- Unmatched GT boxes are false negatives → IoU = 0 ---
    for g_idx in range(len(gt_boxes)):
        if g_idx not in matched_gt:
            ious.append(0)

if ious:
    print(f"Average IoU: {np.mean(ious):.4f}")
    print(f"Total boxes evaluated: {len(ious)}")
else:
    print("No boxes found")

Average IoU: 0.6388
Total boxes evaluated: 270


Demonstrate detection on at least 20 custom real-world test images 

In [ ]:


test_images_path = "/kaggle/working/dataset/test/images"

images = glob.glob(test_images_path + "/*.jpg")
sample_images = random.sample(images, 20)


# results = model.predict(source=sample_images, conf=0.25, save=True)
results = model.predict(
    source=sample_images,
    conf=0.5,     # higher confidence → fewer weak boxes
    iou=0.5,      # stricter NMS → remove overlaps
    save=True
)

output_dir = "/kaggle/working/runs/detect/predict"



pred_images = glob.glob(output_dir + "/*.jpg")

cols = 3   # fewer columns → bigger images
rows = int(np.ceil(len(pred_images[:20]) / cols))

plt.figure(figsize=(20, 6 * rows))

for i, img_path in enumerate(pred_images[:20]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i+1)
    plt.imshow(img)
    plt.axis("off")

plt.tight_layout()
plt.show()

Phase 4 — Model Interpretability

In [ ]:

# ─────────────────────────────────────────
# 1. Load test images
# ─────────────────────────────────────────
test_image_paths = glob.glob("/kaggle/working/dataset/test/images/*.jpg")[:10]

# ─────────────────────────────────────────
# 2. Preprocess
# ─────────────────────────────────────────
def load_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    return (img / 255.0).astype(np.float32)

# ─────────────────────────────────────────
# 3. Find last Conv2D layer name
#    Works for both flat Sequential CNN
#    and nested MobileNetV2 wrappers
# ─────────────────────────────────────────
def get_last_conv_layer_name(model):
    """
    Returns (layer_name, is_nested).
    is_nested=True  → the conv is inside a sub-model (MobileNetV2).
    is_nested=False → the conv is a direct layer of `model`.
    """
    # Search direct layers first (custom CNN)
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name, False

    # Search one level of nesting (MobileNetV2 sub-model)
    for layer in reversed(model.layers):
        if hasattr(layer, 'layers'):
            for sub in reversed(layer.layers):
                if isinstance(sub, tf.keras.layers.Conv2D):
                    return sub.name, True

    raise ValueError("No Conv2D layer found in model")


# ─────────────────────────────────────────
# 4. Grad-CAM  (fixed for Sequential + Functional)
# ─────────────────────────────────────────
def gradcam(img_np, model, conv_name, is_nested):
    img_tensor = tf.constant(np.expand_dims(img_np, 0))  # (1,224,224,3)

    if is_nested:
        # ── MobileNetV2 path (unchanged) ──────────────────────────────
        backbone = None
        for layer in model.layers:
            if hasattr(layer, 'layers'):
                backbone = layer
                break

        conv_layer = backbone.get_layer(conv_name)
        backbone_grad_model = tf.keras.Model(
            inputs=backbone.inputs,
            outputs=[conv_layer.output, backbone.output]
        )

        post_layers = []
        found_backbone = False
        for layer in model.layers:
            if layer is backbone:
                found_backbone = True
                continue
            if found_backbone:
                post_layers.append(layer)

        with tf.GradientTape() as tape:
            conv_out, backbone_out = backbone_grad_model(img_tensor, training=False)
            tape.watch(conv_out)
            x = backbone_out
            for layer in post_layers:
                x = layer(x, training=False)
            preds = x
            class_idx = tf.argmax(preds[0])
            loss = preds[:, class_idx]

        grads = tape.gradient(loss, conv_out)

    else:
        # ── Sequential / Custom CNN path (KEY FIX) ───────────────────
        # Sequential models don't support model.inputs/.output for
        # sub-graph extraction. We manually forward layer-by-layer,
        # capturing the target conv output with a GradientTape watch.

        # Split layers into: before-and-including conv | after conv
        all_layers = model.layers
        conv_idx = next(
            i for i, l in enumerate(all_layers) if l.name == conv_name
        )

        pre_layers  = all_layers[:conv_idx + 1]   # up to & including conv
        post_layers = all_layers[conv_idx + 1:]   # everything after

        # Forward pre-layers to get conv output, then watch it
        x = img_tensor
        for layer in pre_layers:
            x = layer(x, training=False)
        conv_out = x  # shape: (1, H, W, C)

        with tf.GradientTape() as tape:
            tape.watch(conv_out)
            y = conv_out
            for layer in post_layers:
                y = layer(y, training=False)
            preds = y
            class_idx = tf.argmax(preds[0])
            loss = preds[:, class_idx]

        grads = tape.gradient(loss, conv_out)

    # ── Compute heatmap (shared) ──────────────────────────────────────
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))       # (C,)
    conv_out_squeezed = conv_out[0]                             # (H, W, C)
    heatmap = conv_out_squeezed @ pooled_grads[..., tf.newaxis] # (H, W, 1)
    heatmap = tf.squeeze(heatmap)                               # (H, W)
    heatmap = tf.maximum(heatmap, 0)

    max_val = tf.reduce_max(heatmap)
    if max_val == 0:
        return np.zeros((7, 7), dtype=np.float32)

    return (heatmap / max_val).numpy()
# ─────────────────────────────────────────
# 5. Saliency Map
# ─────────────────────────────────────────
def saliency_map(img_np, model):
    """
    img_np : (224,224,3) float32 in [0,1]
    Returns saliency (224,224) float32
    """
    img_tensor = tf.Variable(
        tf.cast(np.expand_dims(img_np, 0), tf.float32)
    )

    with tf.GradientTape() as tape:
        preds = model(img_tensor, training=False)
        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, img_tensor)           # (1,224,224,3)
    sal = tf.reduce_max(tf.abs(grads), axis=-1)[0]   # (224,224)
    return sal.numpy()

# ─────────────────────────────────────────
# 6. Overlay helper
# ─────────────────────────────────────────
def overlay_heatmap(img_np, heatmap):
    """Blends Grad-CAM heatmap onto original image."""
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_uint8  = np.uint8(255 * heatmap_resized)
    heatmap_color  = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color  = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    img_uint8 = np.uint8(img_np * 255)
    overlay   = cv2.addWeighted(img_uint8, 0.55, heatmap_color, 0.45, 0)
    return overlay

# ─────────────────────────────────────────
# 7. Main visualize function
# ─────────────────────────────────────────
def visualize(model, image_paths, class_names=None, num_images=10, model_title="Model"):
    conv_name, is_nested = get_last_conv_layer_name(model)
    print(f"[{model_title}] Using conv layer: '{conv_name}'  (nested={is_nested})")

    n = min(num_images, len(image_paths))
    fig, axes = plt.subplots(n, 3, figsize=(13, 4.5 * n))
    fig.suptitle(f"Grad-CAM & Saliency — {model_title}", fontsize=16, fontweight="bold", y=1.002)

    if n == 1:
        axes = [axes]   # keep indexing consistent

    for i, path in enumerate(image_paths[:n]):
        img = load_image(path)

        # ── Predictions ─────────────────────────────────
        preds     = model.predict(np.expand_dims(img, 0), verbose=0)
        label_idx = int(np.argmax(preds[0]))
        confidence = float(preds[0][label_idx])
        label_str = class_names[label_idx] if class_names else str(label_idx)

        # ── Grad-CAM ─────────────────────────────────────
        heatmap = gradcam(img, model, conv_name, is_nested)
        overlay = overlay_heatmap(img, heatmap)

        # ── Saliency ─────────────────────────────────────
        sal = saliency_map(img, model)

        # ── Plot ─────────────────────────────────────────
        axes[i][0].imshow(img)
        axes[i][0].set_title(f"Original\nPred: {label_str} ({confidence:.1%})", fontsize=9)
        axes[i][0].axis("off")

        axes[i][1].imshow(overlay)
        axes[i][1].set_title("Grad-CAM Overlay", fontsize=9)
        axes[i][1].axis("off")

        axes[i][2].imshow(sal, cmap="hot")
        axes[i][2].set_title("Saliency Map", fontsize=9)
        axes[i][2].axis("off")

    plt.tight_layout()
    plt.show()

# ─────────────────────────────────────────
# 8. Run
# ─────────────────────────────────────────
visualize(mnv2_model,    test_image_paths, class_names=class_names, num_images=10, model_title="MobileNetV2")
visualize(custom_model,  test_image_paths, class_names=class_names, num_images=10, model_title="Custom CNN")